# 흥행 지표 검증

흥행 지표 후보로 `total_reviews`와 `owners_lower`를 선정하고, 두 지표 간의 상관관계를 확인합니다.

- `total_reviews`: Steam 공식 집계 총 리뷰 수 **(주 지표)**
- `owners_lower`: Steam Spy 추정 판매량 하한 **(교차 검증용 보조 지표)**

In [15]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

pd.set_option('display.max_columns', None)

In [16]:
sample_file = "../../../data/processed/steam_stratified_sample.csv"
df_all = pd.read_csv(sample_file)

# large_high, mid_high, small_high 계층만 분석 대상으로 필터링
target_strata = ['large_high', 'mid_high', 'small_high']
df = df_all[df_all['stratum'].isin(target_strata)].copy()

print(f"전체 게임 수: {len(df_all)}")
print(f"분석 대상 게임 수 (high 계층): {len(df)}")
print(f"\n계층별 게임 수:")
print(df['stratum'].value_counts())
df[['name_store', 'stratum', 'owners_lower', 'total_reviews', 'positive', 'negative', 'ccu']].head()


전체 게임 수: 160
분석 대상 게임 수 (high 계층): 74

계층별 게임 수:
stratum
large_high    29
mid_high      25
small_high    20
Name: count, dtype: int64


,name_store,stratum,owners_lower,total_reviews,positive,negative,ccu
0,Sun Haven,large_high,500000,22456,18523,3933,653
1,(the) Gnorp Apologue,large_high,200000,8143,7849,294,289
2,轮回修仙路,large_high,200000,2865,2355,510,14
3,MiSide,large_high,1000000,111087,108883,2204,631
4,Necesse,large_high,1000000,17071,15988,1083,507


In [17]:
# 1. owners_lower vs total_reviews 기초 통계
print("=== 흥행 지표 후보 기초 통계 ===")
print(df[['owners_lower', 'total_reviews', 'ccu']].describe().round(1))

=== 흥행 지표 후보 기초 통계 ===
       owners_lower  total_reviews      ccu
count          74.0           74.0     74.0
mean       239594.6         7741.1   1646.7
std        641467.6        22873.5   9998.9
min             0.0          383.0      0.0
25%             0.0          498.5      2.0
50%         50000.0         1131.0      9.5
75%        200000.0         4893.2     81.5
max       5000000.0       153566.0  83936.0


In [18]:
# 2. owners_lower vs total_reviews Spearman 상관계수
corr, p = stats.spearmanr(df['owners_lower'], df['total_reviews'])
corr_ccu, p_ccu = stats.spearmanr(df['owners_lower'].dropna(), df['ccu'].dropna())

print(f"owners_lower vs total_reviews: r = {corr:.3f}, p = {p:.4f}")
print(f"owners_lower vs ccu:           r = {corr_ccu:.3f}, p = {p_ccu:.4f}")

owners_lower vs total_reviews: r = 0.885, p = 0.0000
owners_lower vs ccu:           r = 0.731, p = 0.0000


In [19]:
# 3. 산점도: owners_lower vs total_reviews
fig = px.scatter(
    df,
    x='owners_lower',
    y='total_reviews',
    text='name_store',
    color='stratum',
    title=f'owners_lower vs total_reviews (Spearman r={corr:.3f}, p={p:.4f})',
    labels={'owners_lower': '추정 판매량 하한 (owners_lower)', 'total_reviews': '총 리뷰 수 (total_reviews)'},
    template='plotly_white'
)
fig.update_traces(textposition='top center')
fig.update_layout(height=600)
fig.show()

In [20]:
# 4. 로그 스케일 산점도 (분포가 우편향이므로 로그 변환 후 확인)
df['log_owners'] = np.log1p(df['owners_lower'])
df['log_reviews'] = np.log1p(df['total_reviews'])

corr_log, p_log = stats.spearmanr(df['log_owners'], df['log_reviews'])

fig_log = px.scatter(
    df,
    x='log_owners',
    y='log_reviews',
    text='name_store',
    color='stratum',
    trendline='ols',
    title=f'[로그 변환] owners_lower vs total_reviews (Spearman r={corr_log:.3f}, p={p_log:.4f})',
    labels={'log_owners': 'log(owners_lower)', 'log_reviews': 'log(total_reviews)'},
    template='plotly_white'
)
fig_log.update_layout(height=600)
fig_log.show()

In [21]:
# 5. 계층(stratum)별 분포 확인 — 지표가 계층을 잘 구분하는지 검증
fig_box = go.Figure()
for col, label in [('owners_lower', '판매량(owners_lower)'), ('total_reviews', '총 리뷰 수')]:
    for stratum in sorted(df['stratum'].unique()):
        fig_box.add_trace(go.Box(
            y=df[df['stratum'] == stratum][col],
            name=f'{stratum}',
            boxmean=True
        ))

fig_owners = px.box(
    df, x='stratum', y='owners_lower',
    title='계층별 owners_lower 분포',
    labels={'owners_lower': '추정 판매량 하한', 'stratum': '계층'},
    color='stratum',
    template='plotly_white'
)
fig_owners.update_layout(height=450)
fig_owners.show()

fig_reviews = px.box(
    df, x='stratum', y='total_reviews',
    title='계층별 total_reviews 분포',
    labels={'total_reviews': '총 리뷰 수', 'stratum': '계층'},
    color='stratum',
    template='plotly_white'
)
fig_reviews.update_layout(height=450)
fig_reviews.show()

In [23]:
# 결론 요약
print("=" * 60)
print("흥행 지표 검증 요약")
print("=" * 60)
print(f"\n[주 지표] total_reviews")
print(f"  - Steam이 직접 집계한 총 리뷰 수")
print(f"  - 범위: {df['total_reviews'].min():,.0f} ~ {df['total_reviews'].max():,.0f}")
print(f"\n[보조 지표] owners_lower (교차 검증용)")
print(f"  - Steam Spy 추정 판매량 하한값")
print(f"  - 범위: {df['owners_lower'].min():,.0f} ~ {df['owners_lower'].max():,.0f}")
print(f"\n[두 지표 간 Spearman 상관계수]")
print(f"  r = {corr:.3f}, p = {p:.4f} {'(유의미)' if p < 0.05 else '(비유의미)'}")
print(f"\n[근거]")
print(f"  - total_reviews는 Steam 공식 집계 수치로 추정값이 아님")
print(f"  - 업계에서 리뷰 수는 판매량의 표준 proxy 지표로 활용됨")
print(f"  - owners_lower와 r={corr:.3f}로 강한 상관 → 두 지표가 같은 흥행을 측정함을 교차 검증")
print(f"  - '왜 이 지표를 썼나'에 대해 Steam 공식 데이터라는 명확한 근거 제시 가능")
print("=" * 60)


흥행 지표 검증 요약

[주 지표] total_reviews
  - Steam이 직접 집계한 총 리뷰 수
  - 범위: 383 ~ 153,566

[보조 지표] owners_lower (교차 검증용)
  - Steam Spy 추정 판매량 하한값
  - 범위: 0 ~ 5,000,000

[두 지표 간 Spearman 상관계수]
  r = 0.885, p = 0.0000 (유의미)

[근거]
  - total_reviews는 Steam 공식 집계 수치로 추정값이 아님
  - 업계에서 리뷰 수는 판매량의 표준 proxy 지표로 활용됨
  - owners_lower와 r=0.885로 강한 상관 → 두 지표가 같은 흥행을 측정함을 교차 검증
  - '왜 이 지표를 썼나'에 대해 Steam 공식 데이터라는 명확한 근거 제시 가능
